# Data Fetching — Bybit (Migrated from MT5)

> **Migration note**: This notebook was migrated from `00_data_feching.ipynb` (MT5 version).
> The execution layer changed from **MetaTrader5** to **Bybit REST API** (`pybit`).
> Strategy logic is unchanged — same symbols, timeframes, and CSV output format.

Fetches OHLCV candles from Bybit Linear (USDT-perpetual) and saves to:
`./data/<SYMBOL>/<TIMEFRAME>/ohlcv.csv`

The CSV output is **column-compatible** with the MT5 version so all downstream notebooks work unchanged.

## Differences from MT5 version
| MT5 | Bybit |
|-----|-------|
| `mt5.copy_rates_from_pos()` | `pybit` kline endpoint |
| Broker symbols (`BTCUSD.i`) | Bybit symbols (`BTCUSDT`) |
| Local terminal required | REST API only |
| `tick_volume` column | `volume` column (base asset qty) |


In [41]:
# SECTION 1 — Imports and parameters
import warnings
warnings.filterwarnings("ignore")

import sys, os, time
from pathlib import Path
import pandas as pd

# Resolve project root (notebooks/ -> parent)
_ROOT = Path.cwd().resolve()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

# Load .env from project root
try:
    from dotenv import load_dotenv
    _env = _ROOT / ".env"
    if _env.exists():
        load_dotenv(_env)
        print(f"Loaded .env from {_env}")
    else:
        print(f"No .env at {_env} — set BYBIT_API_KEY / BYBIT_API_SECRET manually")
except ImportError:
    print("python-dotenv not installed; relying on environment variables")

# ── Fetch parameters ──────────────────────────────────────────────
SYMBOLS = ["DOGEUSD"]
# Other options: "ETHUSDT", "SOLUSDT", "XRPUSDT", "BNBUSDT", "ADAUSDT" , "LNKUSDT" , "BCHUSDT" , "UNIUSDT" , "LTCUSD"
# "DOGEUSD"

TIMEFRAMES = [ "M5","H1","D1"]

BYBIT_INTERVAL_MAP = {
    "M1": "1",  "M3": "3",  "M5": "5",  "M15": "15", "M30": "30",
    "H1": "60", "H2": "120","H4": "240","H6": "360", "H12": "720",
    "D1": "D",  "W1": "W",  "MN": "M",
}

BARS_BY_TF = {
    "M1": 100000, "M5": 100000, "M15":50000, "M30": 50000,
    "H1": 20000,  "H4": 10000,  "D1": 3000,
}

CATEGORY = "linear"
TESTNET  = os.getenv("BYBIT_TESTNET", "false").lower() == "true"

print(f"Symbols   : {SYMBOLS}")
print(f"Timeframes: {TIMEFRAMES}")
print(f"Testnet   : {TESTNET}")


Loaded .env from D:\bot\ema-1d trend-exchange\.env
Symbols   : ['DOGEUSD']
Timeframes: ['M5', 'H1', 'D1']
Testnet   : False


In [42]:
# SECTION 2 — Bybit client setup
from pybit.unified_trading import HTTP

api_key    = os.getenv("BYBIT_API_KEY", "")
api_secret = os.getenv("BYBIT_API_SECRET", "")

client = HTTP(
    testnet=TESTNET,
    api_key=api_key or None,
    api_secret=api_secret or None,
)

print(f"Bybit client ready (testnet={TESTNET})")
print(f"API key set: {bool(api_key)}")


Bybit client ready (testnet=False)
API key set: True


In [43]:
# SECTION 3 — Fetch helpers
def fetch_bybit_klines(symbol: str, interval: str, limit: int = 1000) -> pd.DataFrame:
    """Fetch up to `limit` candles using time-based pagination (end param)."""
    all_rows = []
    end_time  = None
    remaining = limit
    MAX_RETRIES = 5

    while remaining > 0:
        batch  = min(remaining, 1000)
        kwargs = dict(category=CATEGORY, symbol=symbol, interval=interval, limit=batch)
        if end_time is not None:
            kwargs["end"] = end_time

        # retry loop for this batch
        rows = None
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                resp = client.get_kline(**kwargs)
                if resp.get("retCode") != 0:
                    raise RuntimeError(f"Bybit error {resp.get('retCode')}: {resp.get('retMsg')}")
                rows = resp["result"].get("list", [])
                break  # success
            except Exception as exc:
                if attempt == MAX_RETRIES:
                    raise RuntimeError(
                        f"Failed after {MAX_RETRIES} attempts for {symbol} {interval}: {exc}"
                    ) from exc
                wait = 2 ** attempt  # 2, 4, 8, 16 seconds
                print(f"    attempt {attempt}/{MAX_RETRIES} failed ({exc}), retrying in {wait}s...")
                time.sleep(wait)

        if not rows:
            break

        all_rows.extend(rows)
        remaining -= len(rows)
        # oldest timestamp in this batch (rows are newest-first) minus 1ms to avoid overlap
        end_time = int(rows[-1][0]) - 1
        if len(rows) < batch:
            break
        time.sleep(0.05)  # rate-limit safety

    if not all_rows:
        raise RuntimeError(f"No data for {symbol} {interval}")

    df = pd.DataFrame(
        list(reversed(all_rows)),  # oldest-first
        columns=["time", "open", "high", "low", "close", "volume", "turnover"],
    )
    df["time"] = pd.to_datetime(df["time"].astype("int64"), unit="ms", utc=True)
    df = df.set_index("time").sort_index()
    df = df[~df.index.duplicated(keep="last")]  # remove boundary duplicates
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)

    # Drop forming (live) candle — mirrors MT5 behavior
    if len(df) > 0:
        df = df.iloc[:-1]

    return df[["open", "high", "low", "close", "volume"]]


def resolve_bybit_symbol(symbol: str) -> str | None:
    resp = client.get_instruments_info(category=CATEGORY, symbol=symbol)
    if resp.get("retCode") != 0:
        return None
    return symbol if resp.get("result", {}).get("list") else None


print("Fetch helpers defined.")


Fetch helpers defined.


In [44]:
# SECTION 4 — Fetch and cache all symbols / timeframes
BASE_DIR = Path("./data")
results  = {}

for symbol in SYMBOLS:
    resolved = resolve_bybit_symbol(symbol)
    if resolved is None:
        print(f"Skip {symbol}: not found on Bybit {CATEGORY}")
        continue

    print(f"Fetching {symbol}...")

    for tf in TIMEFRAMES:
        interval = BYBIT_INTERVAL_MAP.get(tf)
        if interval is None:
            print(f"  Skip {tf}: no interval mapping")
            continue

        bars = BARS_BY_TF.get(tf, 5000)
        try:
            df = fetch_bybit_klines(symbol, interval, limit=bars)
            out_dir  = BASE_DIR / symbol / tf
            out_dir.mkdir(parents=True, exist_ok=True)
            out_file = out_dir / "ohlcv.csv"
            df.to_csv(out_file)
            results[(symbol, tf)] = len(df)
            print(f"  {tf}: {len(df):,} rows -> {out_file}")
        except Exception as exc:
            print(f"  {tf}: FAILED — {exc}")

print(f"\nDone. Cached {len(results)} time series.")

if results:
    sym, tf = next(iter(results))
    preview = pd.read_csv(BASE_DIR / sym / tf / "ohlcv.csv", index_col=0, parse_dates=True)
    print(f"\nPreview ({sym} {tf}):")
    display(preview.tail(5))


Fetching DOGEUSD...
  M5: 99,999 rows -> data\DOGEUSD\M5\ohlcv.csv
  H1: 10,398 rows -> data\DOGEUSD\H1\ohlcv.csv
  D1: 433 rows -> data\DOGEUSD\D1\ohlcv.csv

Done. Cached 3 time series.

Preview (DOGEUSD M5):


,open,high,low,close,volume
time,,,,,
2026-05-12 11:45:00+00:00,0.10903,0.10916,0.10903,0.10916,219.0
2026-05-12 11:50:00+00:00,0.10916,0.10932,0.10916,0.10932,5.0
2026-05-12 11:55:00+00:00,0.10932,0.10933,0.10932,0.10933,5.0
2026-05-12 12:00:00+00:00,0.10933,0.10933,0.10916,0.10916,109.0
2026-05-12 12:05:00+00:00,0.10916,0.10922,0.10914,0.10922,196.0
